# Build MIMIC-III dataset with hourly data
The raw dataset was built with MIMIC_Extract package: <a href="https://github.com/MLforHealth/MIMIC_Extract" target="_blank">link</a>

# Import packages

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1.Load data
The output of MIMIC_Extract is a .hdf file and contains four tables:
* patients: static demographics, static outcomes
* vitals_labs: time-varying vitals and labs (hourly mean, count and standard deviation)
* vitals_labs_mean: time-varying vitals and labs (hourly mean only)
* interventions: hourly binary indicators for administered interventions

In [2]:
dir = "../data/mimic/"

hdf_file_path = dir + "all_hourly_data.hdf"

df_patients = pd.read_hdf(hdf_file_path, key="patients")
df_vitals_labs = pd.read_hdf(hdf_file_path, key="vitals_labs_mean")
# df_vitals_labs = pd.read_hdf(hdf_file_path, key="vitals_labs")
# df_interventions = pd.read_hdf(hdf_file_path, key="interventions")

# 2.Data processing

## 2.1 Patient
* Extract patients of age > 18 and < 89
* No future readmission
* Select a subset of static features
* Regroup ethnicity

In [3]:
df_patients_proc = df_patients[
    (df_patients["age"] > 18)
    & (df_patients["age"] < 89)
    & (df_patients["readmission_30"] == 0)
]

In [4]:
# Select a subset of static variables

selected_vars = [
    "gender",
    "age",
    "ethnicity",
    "admission_type",
    "los_icu",
    "mort_hosp",
]

df_patients_proc = df_patients_proc[selected_vars]

In [5]:
# Regroup ethnicity

df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].str.split("/").str[0]
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].str.split(" - ").str[0]
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].replace(
    [
        "UNKNOWN",
        "PATIENT DECLINED TO ANSWER",
        "MULTI RACE ETHNICITY",
        "AMERICAN INDIAN",
        "UNABLE TO OBTAIN",
        "PORTUGUESE",
        "CARIBBEAN ISLAND",
        "SOUTH AMERICAN",
        "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER",
        "MIDDLE EASTERN",
    ],
    "OTHER",
)
df_patients_proc["ethnicity"] = df_patients_proc["ethnicity"].replace(
    ["HISPANIC OR LATINO"], "HISPANIC"
)

In [6]:
df_patients_proc.describe(include="all")

,gender,age,ethnicity,admission_type,los_icu,mort_hosp
count,31135,31135.000000,31135,31135,31135.000000,31135.000000
unique,2,NaN,5,3,NaN,NaN
top,M,NaN,WHITE,EMERGENCY,NaN,NaN
freq,17980,NaN,22021,24821,NaN,NaN
mean,NaN,62.348948,NaN,NaN,2.623475,0.088967
std,NaN,16.822707,NaN,NaN,1.975600,0.284701
min,NaN,18.022085,NaN,NaN,0.500000,0.000000
25%,NaN,51.567451,NaN,NaN,1.163432,0.000000
50%,NaN,64.296748,NaN,NaN,1.969213,0.000000
75%,NaN,76.129561,NaN,NaN,3.281829,0.000000


In [7]:
print(f"{len(set(df_patients_proc.index.levels[0]))} patients")
print(f"{len(set(df_patients_proc.index.levels[1]))} hospital admins")
print(f"{len(set(df_patients_proc.index.levels[2]))} icu stays")

34472 patients
34472 hospital admins
34472 icu stays


## 2.2 Vital signs and labs
* Select a subset of variables
* Extract first measure within the defined window and for the selected cohort
* The hour when measure was taken was also included as features
* Remove NAs

In [8]:
# Specify a window size in hours
window_size = 48

vs_labs_vars = [
    "diastolic blood pressure",
    "fraction inspired oxygen",
    "glascow coma scale total",
    "glucose",
    "heart rate",
    "height",
    "mean blood pressure",
    "oxygen saturation",
    "systolic blood pressure",
    "temperature",
    "weight",
    "ph",
]

index = ["subject_id", "hadm_id", "icustay_id"]

In [9]:
df_vitals_labs_proc = df_vitals_labs[vs_labs_vars]

df_vitals_labs_proc = df_vitals_labs_proc[
    (
        df_vitals_labs_proc.index.get_level_values("icustay_id").isin(
            set(df_patients_proc.index.get_level_values("icustay_id"))
        )
    )
    & (df_vitals_labs_proc.index.get_level_values("hours_in") <= window_size)
]

df_vitals_labs_proc.columns = df_vitals_labs_proc.columns.droplevel(
    level=["Aggregation Function"]
)
df_vitals_labs_proc.columns.name = None

In [10]:
# Extract the hour and first measure for each measurement

for col in df_vitals_labs_proc.columns:
    df_col = df_vitals_labs_proc[[col]]
    df_col_first_idx = df_col.groupby(index)[col].apply(lambda x: x.first_valid_index())
    df_col_first = df_col.loc[df_col_first_idx.dropna()].reset_index(
        level="hours_in", inplace=False
    )

    df_col_first.rename(columns={"hours_in": f"hours_in ({col})"}, inplace=True)

    df_patients_proc = pd.merge(
        df_patients_proc, df_col_first, "left", left_index=True, right_index=True
    )

In [11]:
# Remove the entire column if 40% of the data is missing

df_count = df_patients_proc.describe().loc["count", :]
col_to_remove = df_count[df_count <= df_patients_proc.shape[0] * 0.6].index.to_list()

df_patients_proc = df_patients_proc.drop(col_to_remove, axis=1)

In [12]:
# Remove rows that have missing value

df_patients_proc = df_patients_proc.dropna()

In [13]:
df_patients_proc.shape

(15118, 24)

# 3.Save data
* Convert continuous vairables to int or float with 1 decimal place
* Split into train and test sets
* Save to files

In [14]:
# Drop ids
df_patients_proc = df_patients_proc.reset_index(drop=True)

# Remove the underscores from the column names
df_patients_proc.rename(columns=lambda x: x.replace("_", " "), inplace=True)

df_patients_proc.shape

(15118, 24)

In [15]:
# Continuous variables convertion

int_vars = [
    "age",
    "mort hosp",
    "hours in (diastolic blood pressure)",
    "diastolic blood pressure",
    "hours in (glucose)",
    "glucose",
    "hours in (heart rate)",
    "heart rate",
    "hours in (mean blood pressure)",
    "mean blood pressure",
    "hours in (oxygen saturation)",
    "oxygen saturation",
    "hours in (systolic blood pressure)",
    "systolic blood pressure",
    "hours in (temperature)",
    "hours in (weight)",
    "hours in (ph)",
]

float_vars = ["los icu", "temperature", "weight", "ph"]

df_patients_proc[int_vars] = df_patients_proc[int_vars].astype(int)
df_patients_proc[float_vars] = df_patients_proc[float_vars].round(1)

In [16]:
df_patients_proc.head()

,gender,age,ethnicity,admission type,los icu,mort hosp,hours in (diastolic blood pressure),diastolic blood pressure,hours in (glucose),glucose,...,hours in (oxygen saturation),oxygen saturation,hours in (systolic blood pressure),systolic blood pressure,hours in (temperature),temperature,hours in (weight),weight,hours in (ph),ph
0,M,76,WHITE,EMERGENCY,6.1,0,0,39,0,198,...,0,74,0,95,5,36.5,21,106.0,0,7.4
1,F,47,WHITE,EMERGENCY,1.7,0,1,63,5,180,...,0,94,1,116,1,37.4,1,53.6,0,7.5
2,M,41,OTHER,EMERGENCY,5.3,1,0,85,0,129,...,0,98,0,160,0,35.5,37,100.3,4,7.4
3,M,72,WHITE,ELECTIVE,7.6,1,2,72,0,170,...,2,98,2,136,2,35.6,44,95.9,0,7.1
4,F,75,WHITE,ELECTIVE,1.1,0,1,59,0,178,...,1,100,1,121,1,36.0,1,93.3,0,7.4


In [17]:
df_patients_proc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15118 entries, 0 to 15117
Data columns (total 24 columns):
 #   Column                               Non-Null Count  Dtype   
---  ------                               --------------  -----   
 0   gender                               15118 non-null  category
 1   age                                  15118 non-null  int64   
 2   ethnicity                            15118 non-null  object  
 3   admission type                       15118 non-null  category
 4   los icu                              15118 non-null  float64 
 5   mort hosp                            15118 non-null  int64   
 6   hours in (diastolic blood pressure)  15118 non-null  int64   
 7   diastolic blood pressure             15118 non-null  int64   
 8   hours in (glucose)                   15118 non-null  int64   
 9   glucose                              15118 non-null  int64   
 10  hours in (heart rate)                15118 non-null  int64   
 11  heart rate     

In [18]:
# Split into train and test sets

df_train, df_test = train_test_split(
    df_patients_proc,
    test_size=0.3,
    random_state=42,
    stratify=df_patients_proc["mort hosp"],
)

In [19]:
# Save data

df_patients_proc.to_csv(dir + "mimic.csv", index=False)
df_train.to_csv(dir + "mimic_train.csv", index=False)
df_test.to_csv(dir + "mimic_test.csv", index=False)